In [1]:
# Install required libraries silently
!pip install -q ultralytics easyocr opencv-python-headless kagglehub scikit-learn pyyaml

# Check GPU allocation (Crucial for CV projects)
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 36.0 MB/s eta 0:00:00
Sat Sep 26 15:32:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |

In [2]:
import os
import sys
import glob
import logging
import xml.etree.ElementTree as ET
import yaml
import shutil
from pathlib import Path
import kagglehub
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# Enterprise Logging Setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | Lymkit-CV-Pipeline | %(levelname)s | %(message)s'
)
logger = logging.getLogger("ANPR_Workspace")

# Define Workspace Paths
BASE_DIR = Path('/content/anpr_workspace')
YOLO_DATA_DIR = BASE_DIR / 'yolo_format'

# Create standard YOLO directory structure
for split in ['train', 'val']:
    for data_type in ['images', 'labels']:
        (YOLO_DATA_DIR / split / data_type).mkdir(parents=True, exist_ok=True)

logger.info(f"Workspace initialized at {BASE_DIR}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
logger.info("Triggering data ingestion from Kaggle...")
dataset_path = Path(kagglehub.dataset_download("saisirishan/indian-vehicle-dataset"))
logger.info(f"Dataset ingested successfully at: {dataset_path}")

# Summarize Dataset
xml_files = list(dataset_path.rglob("*.xml"))
img_files = list(dataset_path.rglob("*.jpeg")) + list(dataset_path.rglob("*.jpg"))

print("\n" + "="*30)
print("📊 DATASET SUMMARY")
print("="*30)
print(f"Total Images Found:      {len(img_files)}")
print(f"Total Annotations (XML): {len(xml_files)}")

if len(img_files) == 0:
    raise FileNotFoundError("No images found! Check dataset path.")

Using Colab cache for faster access to the 'indian-vehicle-dataset' dataset.

📊 DATASET SUMMARY
Total Images Found:      1644
Total Annotations (XML): 1697


In [4]:
def convert_voc_to_yolo(xml_path: Path, img_width: int, img_height: int) -> list:
    """Converts Pascal VOC bounding boxes to YOLO normalized format."""
    yolo_labels = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        for obj in root.findall('.//object'):
            bndbox = obj.find('bndbox')
            if bndbox is None:
                continue

            xmin = float(bndbox.find('xmin').text)
            ymin = float(bndbox.find('ymin').text)
            xmax = float(bndbox.find('xmax').text)
            ymax = float(bndbox.find('ymax').text)

            # YOLO math: normalized center x, center y, width, height
            x_center = ((xmin + xmax) / 2) / img_width
            y_center = ((ymin + ymax) / 2) / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height

            # Class 0 represents license plate
            yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    except Exception as e:
        logger.warning(f"Failed to parse {xml_path.name}: {e}")

    return yolo_labels

In [5]:
logger.info("Transforming annotations to YOLO format and splitting data...")

# Pair images with their XML annotations
valid_pairs = []
for img_path in img_files:
    xml_path = img_path.with_suffix('.xml')
    if xml_path.exists():
        valid_pairs.append((img_path, xml_path))

# 80/20 Train-Validation Split
train_pairs, val_pairs = train_test_split(valid_pairs, test_size=0.2, random_state=42)

def process_split(pairs, split_name):
    for img_path, xml_path in pairs:
        try:
            tree = ET.parse(xml_path)
            size = tree.getroot().find('size')
            w = int(size.find('width').text)
            h = int(size.find('height').text)

            yolo_labels = convert_voc_to_yolo(xml_path, w, h)

            if not yolo_labels:
                continue

            # Copy image and write label
            shutil.copy(img_path, YOLO_DATA_DIR / split_name / 'images' / img_path.name)

            label_path = YOLO_DATA_DIR / split_name / 'labels' / f"{img_path.stem}.txt"
            with open(label_path, 'w') as f:
                f.write('\n'.join(yolo_labels))

        except Exception as e:
            continue

# Execute the split processing
process_split(train_pairs, 'train')
process_split(val_pairs, 'val')

# Generate data.yaml
yaml_path = YOLO_DATA_DIR / 'data.yaml'
yaml_content = {
    'train': str((YOLO_DATA_DIR / 'train' / 'images').absolute()),
    'val': str((YOLO_DATA_DIR / 'val' / 'images').absolute()),
    'nc': 1,
    'names': ['license_plate']
}

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

logger.info("Data preparation completed. YAML file generated.")

In [6]:
logger.info("Initializing YOLOv8n architecture...")
# Start with pre-trained nano model for fast, accurate training
model = YOLO("yolov8n.pt")

logger.info("Commencing training loop...")
# Train the model
results = model.train(
    data=str(yaml_path),
    epochs=50,             # Increase to 100+ for actual production weights
    imgsz=640,             # Standard YOLO resolution
    batch=16,
    device=0,              # Uses Colab GPU
    project=str(BASE_DIR / "runs"),
    name="lymkit_anpr_model"
)

logger.info("Training concluded. Weights saved to workspace.")

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/anpr_workspace/yolo_format/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=lymkit_anpr_model

In [9]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path

# ==========================================
# 1. LOAD YOUR CUSTOM TRAINED MODEL
# ==========================================
# YOLOv8 saves the best weights from your training run here:
weights_path = "/content/anpr_workspace/runs/lymkit_anpr_model/weights/best.pt"

if not Path(weights_path).exists():
    raise FileNotFoundError(f"Weights not found at {weights_path}. Did the training cell finish?")

logger.info(f"Loading custom ANPR model from {weights_path}...")
model = YOLO(weights_path)

# ==========================================
# 2. IMAGE TESTING FUNCTION
# ==========================================
def test_on_image(image_path: str):
    """Runs inference on a single image and displays it in Colab."""
    logger.info(f"Running inference on image: {image_path}")

    # Run prediction
    results = model.predict(source=image_path, conf=0.5) # Only show boxes with >50% confidence

    # YOLO's plot() function draws the bounding boxes automatically
    annotated_img = results[0].plot()

    # Display in Colab using matplotlib (cv2.imshow crashes Colab)
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# ==========================================
# 3. VIDEO TESTING FUNCTION (PRODUCTION GRADE)
# ==========================================
def test_on_video(input_video_path: str, output_video_path: str = "output_anpr.mp4"):
    """Processes a video frame-by-frame using OpenCV and saves the output."""
    logger.info(f"Processing video: {input_video_path}")

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video {input_video_path}")

    # Get video properties for the output writer
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Initialize OpenCV VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    frame_count = 0
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Run YOLO inference on the current frame
        results = model.predict(frame, conf=0.5, verbose=False)

        # Draw bounding boxes on the frame
        annotated_frame = results[0].plot()

        # Write the annotated frame to our output video
        out.write(annotated_frame)
        frame_count += 1

        # Print progress every 30 frames
        if frame_count % 30 == 0:
            logger.info(f"Processed {frame_count} frames...")

    cap.release()
    out.release()
    logger.info(f"Video processing complete! Saved to {output_video_path}")

# ==========================================
# 4. EXECUTION
# ==========================================
# Upload a test video or image to your Colab environment, then pass the path here:

# Example 1: Test an Image
#test_on_image("/content/sample_car.jpg")

# Example 2: Test a Video
# test_on_video("/content/traffic_footage.mp4", "anpr_results.mp4")

In [10]:
# Video par inference run karne ke liye:
test_on_video("/content/trial_model.mp4", "anpr_results.mp4")

In [11]:
from ultralytics import YOLO

# Apna trained model load karein (path verify kar lein)
model = YOLO("/content/anpr_workspace/runs/lymkit_anpr_model/weights/best.pt")

# Validation dataset par accuracy evaluate karein
print("Checking model accuracy on validation set...")
metrics = model.val()

# Results print karein
print("\n--- ACCURACY METRICS ---")
print(f"mAP@50 (Standard Accuracy): {metrics.box.map50:.3f}")
print(f"mAP@50-95 (Strict Accuracy): {metrics.box.map:.3f}")
print(f"Precision (False Positives kitne kam hain): {metrics.box.p[0]:.3f}")
print(f"Recall (Kitne plates miss nahi hue): {metrics.box.r[0]:.3f}")

Checking model accuracy on validation set...
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1852.0±1178.8 MB/s, size: 67.7 KB)
val: Scanning /content/anpr_workspace/yolo_format/val/labels.cache... 329 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 329/329 115.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 6.5it/s 3.2s
                   all        329        329      0.994      0.997      0.994      0.862
Speed: 0.7ms preprocess, 3.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val

--- ACCURACY METRICS ---
mAP@50 (Standard Accuracy): 0.994
mAP@50-95 (Strict Accuracy): 0.862
Precision (False Positives kitne kam hain): 0.994
Recall (Kitne plates miss nahi hue): 0.997


In [12]:
import cv2
from ultralytics import YOLO

model = YOLO("/content/anpr_workspace/runs/lymkit_anpr_model/weights/best.pt")

cap = cv2.VideoCapture("/content/trial_model.mp4")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter("anpr_debug_results.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print("Processing video with aggressive detection settings...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # conf=0.15 aur imgsz=1280 use kar rahe hain chhoti plates pakadne ke liye
    results = model.predict(frame, conf=0.15, imgsz=1280, verbose=False)
    annotated_frame = results[0].plot()
    out.write(annotated_frame)

cap.release()
out.release()
print("Done! Check 'anpr_debug_results.mp4' in left files panel.")

Processing video with aggressive detection settings...
Done! Check 'anpr_debug_results.mp4' in left files panel.
